# Day 5 — Supervised-Learning Mini-Project

# The Full Supervised-Learning Pipeline

A supervised-learning project follows a complete workflow:

1. Explore the data.
2. Preprocess the data.
3. Split the data into training and testing sets.
4. Train machine learning models.
5. Evaluate and compare the models.

The goal is to build a complete and reliable machine learning workflow.

In [17]:
import pandas as pd

# Load the dataset
df = pd.read_csv("dataset.csv")

# Display the first rows
print(df.head())

# Display dataset information
print("\nDataset Information:")
print(df.info())

# Display basic statistics
print("\nDataset Statistics:")
print(df.describe())

   Age  Income  StudyHours      City  Passed
0   18    2500           2    Hebron       0
1   19    3000           4    Nablus       1
2   20    3500           5  Ramallah       1
3   21    2200           1    Hebron       0
4   22    4000           6  Ramallah       1

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Age         20 non-null     int64
 1   Income      20 non-null     int64
 2   StudyHours  20 non-null     int64
 3   City        20 non-null     str  
 4   Passed      20 non-null     int64
dtypes: int64(4), str(1)
memory usage: 932.0 bytes
None

Dataset Statistics:
             Age       Income  StudyHours     Passed
count  20.000000    20.000000   20.000000  20.000000
mean   20.950000  3395.000000    4.450000   0.600000
std     1.877148   872.066511    2.350252   0.502625
min    18.000000  2200.000000    1.000000   0.000000
25

# Handling Missing Values

Missing values can cause problems when training machine learning models.

First, we check the dataset for missing values.

For numerical features, missing values can be replaced with the mean value.

In [18]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Fill missing numerical values with the mean
numeric_columns = df.select_dtypes(include="number").columns

df[numeric_columns] = df[numeric_columns].fillna(
    df[numeric_columns].mean()
)

print("\nMissing values after handling:")
print(df.isnull().sum())

Missing values:
Age           0
Income        0
StudyHours    0
City          0
Passed        0
dtype: int64

Missing values after handling:
Age           0
Income        0
StudyHours    0
City          0
Passed        0
dtype: int64


# One-Hot Encoding

Machine learning models work with numerical data.

Categorical features such as `City` contain text values, so they must be converted into numerical values.

One-hot encoding creates separate columns for the different categories.

In [19]:
# Separate features and target
X = df.drop("Passed", axis=1)
y = df["Passed"]

# One-hot encode the City column
X = pd.get_dummies(
    X,
    columns=["City"],
    drop_first=True
)

print("Encoded Features:")
print(X.head())

Encoded Features:
   Age  Income  StudyHours  City_Nablus  City_Ramallah
0   18    2500           2        False          False
1   19    3000           4         True          False
2   20    3500           5        False           True
3   21    2200           1        False          False
4   22    4000           6        False           True


# Train/Test Split

The dataset is divided into training and testing sets.

The training set is used to train the models.

The testing set contains unseen data and is used to evaluate the models fairly.

We use an 80/20 split and a fixed `random_state` to make the results reproducible.

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (16, 5)
X_test : (4, 5)
y_train: (16,)
y_test : (4,)


# Feature Scaling

Feature scaling puts numerical features on a similar scale.

Scaling is especially important for models that depend on distance or margins, such as:

- k-NN
- SVM

The scaler must be fitted using the training data only.

The test data should only be transformed using the already fitted scaler.

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape :", X_test_scaled.shape)

X_train_scaled shape: (16, 5)
X_test_scaled shape : (4, 5)


# Avoiding Data Leakage

Data leakage happens when information from the test set is used during training.

To prevent this, the scaler is fitted only on the training data.

The test data must only be transformed after the scaler has been fitted.

In [22]:
# Correct approach

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# Choosing the Model and Metric

This dataset is a classification problem because the target `Passed` contains class labels.

We can use classification models such as Logistic Regression and Random Forest.

The F1-score is used to evaluate the models because it combines Precision and Recall into one metric.

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Create the models
logistic_model = LogisticRegression(max_iter=1000)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train Logistic Regression using scaled data
logistic_model.fit(
    X_train_scaled,
    y_train
)

# Train Random Forest using original data
random_forest_model.fit(
    X_train,
    y_train
)

# Make predictions
logistic_predictions = logistic_model.predict(
    X_test_scaled
)

random_forest_predictions = random_forest_model.predict(
    X_test
)

print("Logistic Regression Predictions:")
print(logistic_predictions)

print("\nRandom Forest Predictions:")
print(random_forest_predictions)

Logistic Regression Predictions:
[0 0 1 1]

Random Forest Predictions:
[0 0 1 1]


# Model Evaluation

The models are evaluated using the F1-score.

The F1-score considers both Precision and Recall.

A higher F1-score means better classification performance.

In [24]:
from sklearn.metrics import f1_score

logistic_f1 = f1_score(
    y_test,
    logistic_predictions
)

random_forest_f1 = f1_score(
    y_test,
    random_forest_predictions
)

print("Logistic Regression F1:", logistic_f1)
print("Random Forest F1:", random_forest_f1)

Logistic Regression F1: 1.0
Random Forest F1: 1.0


# Comparing Against a Baseline

A baseline is a simple prediction used as a reference.

For classification, we can use the majority class as the baseline prediction.

The machine learning model should perform better than this simple baseline to show that it has learned useful patterns.

In [25]:
from sklearn.metrics import f1_score

# Find the majority class in the training data
majority_class = y_train.mode()[0]

# Predict the majority class for every test sample
baseline_predictions = [majority_class] * len(y_test)

# Calculate baseline F1-score
baseline_f1 = f1_score(
    y_test,
    baseline_predictions
)

print("Baseline F1:", baseline_f1)
print("Logistic Regression F1:", logistic_f1)
print("Random Forest F1:", random_forest_f1)

Baseline F1: 0.6666666666666666
Logistic Regression F1: 1.0
Random Forest F1: 1.0


# Model Comparison

The models are compared using the same test set and the same F1-score metric.

The model with the highest F1-score performs best on this dataset.

The baseline is also included to determine whether the models add value.

In [26]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "F1 Score": [
        baseline_f1,
        logistic_f1,
        random_forest_f1
    ]
})

print(results)

                 Model  F1 Score
0             Baseline  0.666667
1  Logistic Regression  1.000000
2        Random Forest  1.000000


# Selecting the Better Model

The best model is the one with the highest F1-score.

We compare the two machine learning models and select the model that performs best on the test data.

In [27]:
model_results = results[
    results["Model"] != "Baseline"
]

best_model = model_results.loc[
    model_results["F1 Score"].idxmax()
]

print("Best Model:")
print(best_model)

Best Model:
Model       Logistic Regression
F1 Score                    1.0
Name: 1, dtype: object


# Random Forest Feature Importance

Random Forest provides feature importance scores.

These scores show which features contributed most to the model's predictions.

A higher importance value means the feature had a stronger influence on the model.

In [28]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

         Feature  Importance
2     StudyHours    0.420385
1         Income    0.377554
4  City_Ramallah    0.106295
0            Age    0.093948
3    City_Nablus    0.001818


# Final Results

This project followed a complete supervised-learning pipeline.

The dataset was first explored and checked for missing values.

Categorical data was converted using one-hot encoding.

The data was then divided into training and testing sets.

Feature scaling was applied only after the split to avoid data leakage.

Two classification models were trained:
- Logistic Regression
- Random Forest

The models were evaluated using the F1-score and compared with a baseline.

The model with the highest F1-score was selected as the better model.

In [29]:
print("===== Final Results =====")

print("\nBaseline F1:")
print(baseline_f1)

print("\nLogistic Regression F1:")
print(logistic_f1)

print("\nRandom Forest F1:")
print(random_forest_f1)

print("\nSelected Model:")
print(best_model["Model"])

print("\nSelected Model F1:")
print(best_model["F1 Score"])

===== Final Results =====

Baseline F1:
0.6666666666666666

Logistic Regression F1:
1.0

Random Forest F1:
1.0

Selected Model:
Logistic Regression

Selected Model F1:
1.0


# Hands-On Lab: End-to-End Mini-Project

# Step 1: Choose the Dataset and Determine the Task

The dataset contains information about students, including age, income, study hours, and city.

The target variable is `Passed`.

Because the target represents classes (`0` and `1`), this is a classification problem.

In [30]:
import pandas as pd

df = pd.read_csv("dataset.csv")

print(df.head())

print("\nTarget Distribution:")
print(df["Passed"].value_counts())

   Age  Income  StudyHours      City  Passed
0   18    2500           2    Hebron       0
1   19    3000           4    Nablus       1
2   20    3500           5  Ramallah       1
3   21    2200           1    Hebron       0
4   22    4000           6  Ramallah       1

Target Distribution:
Passed
1    12
0     8
Name: count, dtype: int64


# Step 2: Perform EDA and Preprocess the Data

First, inspect the dataset and check for missing values.

Then handle missing values, encode categorical features, and split the data.

Feature scaling will be fitted only on the training data to prevent data leakage.

In [31]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Separate features and target
X = df.drop("Passed", axis=1)
y = df["Passed"]

# Encode categorical feature
X = pd.get_dummies(
    X,
    columns=["City"],
    drop_first=True
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale the data for Logistic Regression
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Missing Values:
Age           0
Income        0
StudyHours    0
City          0
Passed        0
dtype: int64

Training shape: (16, 5)
Testing shape: (4, 5)


# Step 3: Train and Evaluate at Least Two Models

Train at least two appropriate classification models.

Evaluate both models using the same metric and compare their performance against a baseline.

In [32]:
# Create models
logistic_model = LogisticRegression(max_iter=1000)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train models
logistic_model.fit(
    X_train_scaled,
    y_train
)

random_forest_model.fit(
    X_train,
    y_train
)

# Predictions
logistic_predictions = logistic_model.predict(
    X_test_scaled
)

random_forest_predictions = random_forest_model.predict(
    X_test
)

# F1 scores
logistic_f1 = f1_score(
    y_test,
    logistic_predictions
)

random_forest_f1 = f1_score(
    y_test,
    random_forest_predictions
)

print("Logistic Regression F1:", logistic_f1)
print("Random Forest F1:", random_forest_f1)

Logistic Regression F1: 1.0
Random Forest F1: 1.0


# Step 4: Select the Better Model

The models are compared using their F1-scores.

The model with the highest F1-score is selected as the better model.

The choice is based on the model's performance on unseen test data.

In [33]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "F1 Score": [
        logistic_f1,
        random_forest_f1
    ]
})

print(results)

best_model = results.loc[
    results["F1 Score"].idxmax()
]

print("\nBest Model:")
print(best_model["Model"])

print("F1 Score:")
print(best_model["F1 Score"])

                 Model  F1 Score
0  Logistic Regression       1.0
1        Random Forest       1.0

Best Model:
Logistic Regression
F1 Score:
1.0


# Step 5: Final Documentation

The complete machine learning pipeline was implemented from data exploration to model evaluation.

The dataset was identified as a classification problem.

Missing values were checked, categorical data was encoded, and the dataset was split into training and testing sets.

Feature scaling was performed using only the training data to avoid data leakage.

Two classification models were trained and evaluated using F1-score.

The final model was selected based on its performance on the test set.

In [34]:
print("===== Mini-Project Summary =====")

print("Task: Classification")
print("Metric: F1 Score")

print("\nModel Results:")
print(results)

print("\nBest Model:", best_model["Model"])
print("Best F1 Score:", best_model["F1 Score"])

===== Mini-Project Summary =====
Task: Classification
Metric: F1 Score

Model Results:
                 Model  F1 Score
0  Logistic Regression       1.0
1        Random Forest       1.0

Best Model: Logistic Regression
Best F1 Score: 1.0
